In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M15.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7772011773587252, 'n_it': 0.39267002624416675}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[14.05803462201884, 13.594664519552367, 15.560615535085923, 14.656879300659176, 15.557643255370982, 16.333999110792995, 14.360541928345215, 14.859389455706607, 13.91587256301517, 14.400945110535734, 13.419931518294112, 15.343755454690092, 13.67366142909563, 14.72815602428626, 13.758243466710386, 17.24926548946017, 13.896301883102609, 15.517505024063194, 13.621323596273694, 13.710094767816926, 14.442937585176209, 13.707829488477362, 14.174179974718017, 15.809886827289029, 13.869812089017989, 15.671642204180952, 14.781981016066277, 13.673395641381406, 13.894411565991732, 13.681698708420864, 13.627261714766707, 13.70115300221611, 15.05267115937265, 17.319058524431156, 16.559525966596752, 16.912718942869496, 15.90643503592718, 13.543999361374983, 14.93849163056215, 14.690691432948816, 15.529312746700274, 15.49856224570504, 16.491146733698873, 15.434087021067342, 15.311740027671654, 15.28655618172961, 16.624590181616664, 14.543513372743316, 13.749583802061725, 16.908457332905062, 13.6054368

In [5]:
np.average(y_max_arr)

np.float64(14.94850904073176)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)